# DAEDALUS Colab Training Environment (GRPO)

This notebook runs side-by-side with your Hugging Face Space job. It includes the exact same fixes (`args=config`, `use_gradient_checkpointing=True`, `is_bfloat16_supported`) applied to the cloud cluster.

**Hardware Recommendation:** Go to `Runtime > Change runtime type` and select **T4 GPU** or **A100 GPU**.

In [ ]:
!pip uninstall -y vllm\n!pip install -q huggingface_hub trl unsloth transformers datasets accelerate openenv fastapi pydantic\n!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os
import sys

# 1. Clone our custom DAEDALUS environment codebase straight from your HF Space!
!rm -rf daedalus_repo
!git clone https://REPLACED_WITH_HF_TOKEN_ENV_VAR@huggingface.co/spaces/kabilesh-c/daedalus daedalus_repo

sys.path.insert(0, os.path.abspath('./daedalus_repo'))
print("Environment synced successfully!")

In [ ]:
import json
import random
import torch
import numpy as np
from datasets import Dataset
from typing import List, Dict, Any
from huggingface_hub import login

# Authenticate with your HF token
login(token=os.environ.get("HF_TOKEN"))

from daedalus.env import DaedalusEnvironment
from daedalus.models import Observation

def _format_prompt(obs: dict) -> str:
    lines = [
        "You are a mechanism designer for a market auction system.",
        "Analyze the current market state and design an optimal mechanism.",
        "",
        f"Round: {obs.get('round_number', 0)} / {obs.get('episode_length', 50)}",
        "",
        "Your goal is to maximize the composite reward R = W \u00d7 F \u00d7 P \u00d7 S"
    ]
    outcomes = obs.get('market_outcomes', [])
    if outcomes:
        lines.append("Recent Market Outcomes:")
        for o in outcomes[-5:]:
            lines.append(
                f"  W={o.get('welfare_ratio', 0):.3f} "
                f"F={1-o.get('gini_coefficient', 0):.3f} "
                f"P={o.get('participation_rate', 1):.3f} "
                f"R={o.get('composite_reward', 0):.3f}"
            )
    lines.extend([
        "",
        "Respond with ONLY a JSON mechanism configuration:",
        '{"auction_type": "first_price|second_price|vcg", "reserve_price": float(0-0.9), '
        '"reveal_reserve": bool, "reveal_competing_bids": bool, "reveal_winner_identity": bool, '
        '"reveal_clearing_price": bool, "reveal_bid_distribution": bool, '
        '"shill_penalty": float(0-3), "withdrawal_penalty": float(0-3), "collusion_penalty": float(0-3), '
        '"coalition_policy": "allow|restrict|penalize_suspected|penalize_confirmed"}',
    ])
    return "\n".join(lines)

def _random_mechanism() -> dict:
    return {
        "auction_type": random.choice(["first_price", "second_price", "vcg"]),
        "reserve_price": random.uniform(0.05, 0.5),
        "reveal_reserve": random.random() > 0.5,
        "reveal_competing_bids": random.random() > 0.7,
        "reveal_winner_identity": random.random() > 0.5,
        "reveal_clearing_price": random.random() > 0.3,
        "reveal_bid_distribution": random.random() > 0.7,
        "shill_penalty": random.uniform(0, 2),
        "withdrawal_penalty": random.uniform(0, 1),
        "collusion_penalty": random.uniform(0, 2),
        "coalition_policy": random.choice(["allow", "restrict", "penalize_suspected"]),
    }

def generate_training_prompts(n_prompts: int = 200) -> List[Dict[str, str]]:
    print(f"\nGenerating {n_prompts} training prompts...")
    prompts = []
    env = DaedalusEnvironment()
    for i in range(n_prompts):
        obs_dict = env.reset()
        prompts.append({"prompt": _format_prompt(obs_dict)})
        for step in range(random.randint(1, 5)):
            obs_dict, _, done, _ = env.step(_random_mechanism())
            if done: break
            prompts.append({"prompt": _format_prompt(obs_dict)})
    return prompts

eval_env = DaedalusEnvironment()

def evaluate_mechanism(generated_text: str, env: DaedalusEnvironment) -> float:
    try:
        j_start = generated_text.find('{')
        j_end = generated_text.rfind('}') + 1
        if j_start >= 0 and j_end > j_start:
            mech = json.loads(generated_text[j_start:j_end])
        else:
            return -0.5
    except:
        return -0.5
    _, reward, _, _ = env.step(mech)
    return reward

def grpo_reward_fn(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    rewards = []
    for comp in completions:
        eval_env.reset()
        rewards.append(evaluate_mechanism(comp, eval_env))
    return rewards

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer

print("Loading Dataset...")
dataset = Dataset.from_list(generate_training_prompts(200))

print("Loading Unsloth Qwen2.5-7B...")
# Auto-detects float16 for Colab T4 vs bfloat16 for Colab A100
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=dtype,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing=True, # <-- Critical Fix for GRPO Mismatch!
)

config = GRPOConfig(
    output_dir="./daedalus-colab-checkpoints",
    num_train_epochs=1,                 # <--- Reduced for extreme speed
    per_device_train_batch_size=1,\n    gradient_accumulation_steps=16,\n    num_generations=4,\n    max_completion_length=150,          # <--- Reduced from 512. The JSON is only ~100 tokens long!
    use_vllm=False,                     # <--- Disabled for compatibility\n    learning_rate=5e-6,
    logging_steps=10,
    save_steps=200,
    bf16=is_bfloat16_supported(), # Automatically matches Colab GPU type
    fp16=not is_bfloat16_supported(),
    push_to_hub=True,
    hub_model_id="kabilesh-c/daedalus-designer",
    hub_strategy="every_save",
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    args=config, # <-- Critical API Fix for new TRL version
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    train_dataset=dataset,
)

print("\n\ud83d\ude80 Starting GRPO Training in Colab!...")
trainer.train()

In [ ]:
print("Pushing final adapter to Hugging Face...")
model.push_to_hub("kabilesh-c/daedalus-designer")
tokenizer.push_to_hub("kabilesh-c/daedalus-designer")
print("\u2705 Model successfully deployed to HF Hub from Colab!")